Foundations of Data Science

In [2]:
import sys
import math
import io
import sqlite3
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, mean_squared_error)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.neural_network import MLPClassifier
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print({"python": sys.version.split()[0], "numpy": np.__version__,
       "pandas": pd.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__})

{'python': '3.14.4', 'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'sklearn': '1.9.1'}


In [3]:
usage_list = [100, 200, 300]
usage = np.array(usage_list, dtype=float)
def bill(kwh, rate=0.30, fixed=20):
    return kwh * rate + fixed
print("list repetition:", usage_list * 2)
print("array multiplication:", usage * 2)
print("fare:", bill(usage))
print("200 above:", usage[usage >= 200])

list repetition: [100, 200, 300, 100, 200, 300]
array multiplication: [200. 400. 600.]
fare: [ 50.  80. 110.]
200 above: [200. 300.]


In [4]:
small = pd.DataFrame({"id": [1, 2, 3], "plan": ["A", "B", "A"], "kwh": usage})
small["bill"] = bill(small["kwh"])
print(small)
print(small.loc[small["kwh"] >= 200, ["id", "bill"]])
print(small.groupby("plan")["bill"].agg(["size", "mean"]))

   id plan    kwh   bill
0   1    A  100.0   50.0
1   2    B  200.0   80.0
2   3    A  300.0  110.0
   id   bill
1   2   80.0
2   3  110.0
      size  mean
plan            
A        2  80.0
B        1  80.0


In [6]:
omega = set(range(1, 7))
A, B = {2, 4, 6}, {4, 5, 6}
prob = lambda event: len(event) / len(omega)
print("P(A), P(B):", prob(A), prob(B))
print("intersection:", A & B, prob(A & B))
print("union:", A | B, prob(A | B))
assert np.isclose(prob(A | B), prob(A) + prob(B) - prob(A & B))

P(A), P(B): 0.5 0.5
intersection: {4, 6} 0.3333333333333333
union: {2, 4, 5, 6} 0.6666666666666666


In [7]:
# Of 1,000 customers, 200 complain and 60 of them churn. Of 800 others, 40 churn.
counts = pd.DataFrame({"churn": [60, 40], "stay": [140, 760]}, index=["complaint", "no_complaint"])
print(counts)
p_churn = counts["churn"].sum() / counts.to_numpy().sum()
p_churn_given_complaint = 60 / 200
p_complaint_given_churn = 60 / 100
print(p_churn, p_churn_given_complaint, p_complaint_given_churn)
print("Independent?", np.isclose(p_churn, p_churn_given_complaint))

              churn  stay
complaint        60   140
no_complaint     40   760
0.1 0.3 0.6
Independent? False


In [8]:
p_churn_given_wiout_complaint = 40 / 800
print(p_churn_given_wiout_complaint)

0.05


In [9]:
prior, sensitivity, false_positive = 0.10, 0.80, 0.20
true_alerts = 1000 * prior * sensitivity
false_alerts = 1000 * (1 - prior) * false_positive
posterior = true_alerts / (true_alerts + false_alerts)
print({"true_alerts": true_alerts, "false_alerts": false_alerts, "posterior": posterior})

{'true_alerts': 80.0, 'false_alerts': 180.0, 'posterior': 0.3076923076923077}
